# Train a model on Colab

1. **Runtime → Change runtime type → A100 GPU** (or L4 to use fewer compute units). Not CPU or TPU.
2. Run the cells in order. Keep this tab open while training.
3. If Colab disconnects: reconnect, run **Setup** again, then the same **Train** cell. Training **resumes** from the last checkpoint, which is kept in Google Drive.
4. When you're done: **Runtime → Disconnect and delete runtime**, so it stops using compute units.

Edit code on your laptop and push to GitHub; changes made here are lost.

## Setup

In [ ]:
BRANCH = "main"
REPO = "https://github.com/alliemi/cis5300-finalproject.git"

import os
os.chdir("/content")
if not os.path.exists("cis5300-finalproject"):
    !git clone -b {BRANCH} {REPO}
%cd /content/cis5300-finalproject
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!test -f data/train.jsonl || tar -xzf data.tar.gz

# Keep runs/ (checkpoints, logs, predictions, scores) in Google Drive so they survive disconnects.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cis5300-runs && ln -sfn /content/drive/MyDrive/cis5300-runs runs
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Choose the run

**Every model must use the same `FLAGS`**, so that only the attention differs. Agree on them as a team and don't change them between models.

In [ ]:
ATTENTION = "linear"     # softmax | linear
SEED = 0
FLAGS = "--curriculum-steps 4000"     # the team's shared recipe
RUN = f"runs/{ATTENTION}-s{SEED}"
print(RUN)

## Timing run (~1–2 min)

200 steps to measure speed on this GPU, using all lengths (curriculum off, so the speed is representative). The full run is `--steps` (default 20,000) steps.

In [ ]:
!python train.py --attention {ATTENTION} --seed {SEED} {FLAGS} --curriculum-steps 0 --out {RUN}-timing --steps 200 --eval-every 200 --no-final-eval

## Train

When it finishes, it predicts the full dev split with the best checkpoint and writes `dev_pred.jsonl` and `dev_scores.json` (via `score.py`) into the run folder. Rerunning this cell resumes an interrupted run.

In [ ]:
!python train.py --attention {ATTENTION} --seed {SEED} {FLAGS} --out {RUN}

## Compare finished runs

In [ ]:
!python summarize_runs.py